# UNIFICACION DE DATOS

In [1]:
#trataniento de datos
import pandas as pd

pd.set_option('display.max_columns',None)

In [2]:
df_raw_bank = pd.read_csv('../Datos.Originales/bank-additional.csv')
df_raw_customer = pd.read_excel('../Datos.Originales/customer-details.xlsx')

In [3]:
df_bank=df_raw_bank.copy()
df_customer=df_raw_customer.copy()


In [4]:
nombres_hojas = pd.ExcelFile('../Datos.Originales/customer-details.xlsx')
print(nombres_hojas.sheet_names)

['2012', '2013', '2014']


In [5]:
df_c2012 = pd.read_excel('../Datos.Originales/customer-details.xlsx', sheet_name='2012')
df_c2013 = pd.read_excel('../Datos.Originales/customer-details.xlsx', sheet_name='2013')
df_c2014 = pd.read_excel('../Datos.Originales/customer-details.xlsx', sheet_name='2014')

In [6]:
df_bank.columns

Index(['Unnamed: 0', 'age', 'job', 'marital', 'education', 'default',
       'housing', 'loan', 'contact', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx',
       'cons.conf.idx', 'euribor3m', 'nr.employed', 'y', 'date', 'latitude',
       'longitude', 'id_'],
      dtype='object')

In [7]:
df_c2012.columns

Index(['Unnamed: 0', 'Income', 'Kidhome', 'Teenhome', 'Dt_Customer',
       'NumWebVisitsMonth', 'ID'],
      dtype='object')

In [8]:
df_c2013.columns

Index(['Unnamed: 0', 'Income', 'Kidhome', 'Teenhome', 'Dt_Customer',
       'NumWebVisitsMonth', 'ID'],
      dtype='object')

In [9]:
df_c2014.columns

Index(['Unnamed: 0', 'Income', 'Kidhome', 'Teenhome', 'Dt_Customer',
       'NumWebVisitsMonth', 'ID'],
      dtype='object')

* Vemos que los nombres de las columnas de cada hoja del archivo de excel tienen los mismos nombres por lo que podremos unificarlas verticalemente en una misma hoja. Una de las columnas muestra la fecha, así que al unificar las hojas no perderemos la informacion relativa al año.

* Al ver que los archivos csv y excel tienen una columna ID podemos intuir que puede referirsse al mismo ID en ambos archivos. Este puede ser un nexo de unión para los dos archivos.
A continuacion comprobamos si todos los elementos de la columna ID de df_customer estan en df_bank 

In [10]:
ids_bank = set(df_bank['id_'])
ids_customer = set(df_c2012['ID']).union(df_c2013['ID'], df_c2014['ID'])

In [11]:
ids_no_en_customer = ids_bank - ids_customer
ids_no_en_customer_series = pd.Series(list(ids_no_en_customer))
Num_ids_no_en_customer = len(ids_no_en_customer_series)
Num_ids_no_en_customer

0

In [12]:
ids_no_en_bank = ids_customer - ids_bank
ids_no_en_bank_series = pd.Series(list(ids_no_en_bank))
Num_ids_no_en_bank = len(ids_no_en_bank_series)
Num_ids_no_en_bank

170

* Hay 170 IDs que estan en customer pero no en bank. La prioridad en este caso es tener todos los uids de Bank cubiertos con la informacion adicional proporcionada por customer. Aqui se cumple esto asi que podremos transferir la informacion de customer al archivo csv. 

## UNIFICACION DE LAS TRES HOJAS EN UNA SOLA (XLSX)

In [13]:
df_años_unificado = pd.concat([df_c2012,df_c2013,df_c2014], ignore_index=True)

* Tras la unificacion de las tres hojas realizamos de nuevo la comprobacion de los IDs entre el df_bank y el df_años_unificado

In [14]:
ids_bank = set(df_bank['id_'])
ids_unificados = set(df_años_unificado['ID'])

In [15]:
ids_no_en_unificados = ids_bank - ids_unificados
ids_no_en_unificados_series = pd.Series(list(ids_no_en_unificados))
Num_ids_no_en_unificados = len(ids_no_en_unificados_series)
Num_ids_no_en_unificados

0

In [16]:
ids_no_en_bank = ids_unificados - ids_bank
ids_no_en_bank_series = pd.Series(list(ids_no_en_bank))
Num_ids_no_en_bank = len(ids_no_en_bank_series)
Num_ids_no_en_bank

170

## UNIFICACION DEL CSV Y EL XLSX (en un CSV)

In [17]:
# Contar duplicados en df_c['id_']
num_duplicados_df_bank = df_bank['id_'].duplicated(keep=False).sum()
print(f"Número de duplicados en df_bank['id']: {num_duplicados_df_bank}")

# Contar duplicados en df_años_unificado['ID']
num_duplicados_df_años_unificado = df_años_unificado['ID'].duplicated(keep=False).sum()
print(f"Número de duplicados en df_años_unificado['ID']: {num_duplicados_df_años_unificado}")

Número de duplicados en df_bank['id']: 0
Número de duplicados en df_años_unificado['ID']: 0


* No hay duplicados en ninguno de los dos ID's asi que procedemos a la union de dos df

In [18]:
df_unidos =pd.merge(df_bank,df_años_unificado, left_on='id_', right_on='ID', how='left')

* guardamos el nuevo data frame unificado a un nuevo csv

In [19]:
df_unidos.to_csv('../Datos/df_data_unificada.csv', index=False)